# PPO θ.4 — true PFSP self-history training on Colab Pro+ A100

> **Migration trigger**: WSL2 15 GB RAM で OOM (2026-05-11 16:22 JST)、 Colab Pro+ へ移行
> **Reference**: `docs/dev/remote-migration-2026-05-11.md`

## 必要な事前準備 (= user 手動)

1. Colab Pro+ アカウントで本 notebook を開く
2. Runtime → Change runtime type → **A100 GPU** + **High-RAM** を選択
3. Google Drive を mount (= warm-start zip と output 永続化)
4. GitHub repo に push 済の最新 main を pull-only で取得 (= secrets 排除)
5. `agents/proxy/ppo_v3_theta3.zip` を Drive `/MyDrive/orbit-wars/ppo_v3_theta3.zip` に予め配置

## Cell 1: Repo clone + Drive mount

In [ ]:
import os
import subprocess

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/<USER>/orbit-wars.git"  # public mirror or PAT-authed URL
REPO_DIR = "/content/orbit-wars"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("repo at", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"])

## Cell 2: Dependencies install

Colab base image は Python 3.10、 sb3-contrib + kaggle_environments を追加。

In [ ]:
!pip install -q sb3-contrib==2.5.0 stable-baselines3==2.5.0 kaggle_environments gymnasium==0.29.1
!pip install -q numpy torch  # already on Colab but ensure version
!python -c 'import torch; print("cuda:", torch.cuda.is_available(), "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")'

## Cell 3: PFSP training 起動

- `--total-timesteps 200000` (= local 50k の 4 倍、 A100 で 4-6 hour 想定)
- `--n-envs 8` (= High-RAM 51 GiB なら余裕)
- `--warm-start /content/drive/MyDrive/orbit-wars/ppo_v3_theta3.zip` (= 既存 θ.3 zip)
- output を Drive に直接書く

In [ ]:
import datetime
import os
import subprocess

WARM_START = "/content/drive/MyDrive/orbit-wars/ppo_v3_theta3.zip"
OUTPUT = "/content/drive/MyDrive/orbit-wars/ppo_v4_theta4.zip"
POOL_DIR = "/content/drive/MyDrive/orbit-wars/ppo_pfsp_pool_theta4"
LOG = (
    '/content/drive/MyDrive/orbit-wars/'
    f'ppo_v4_theta4_{datetime.datetime.now().strftime("%Y%m%dT%H%M")}.log'
)

os.makedirs(POOL_DIR, exist_ok=True)

cmd = [
    "python",
    "-m",
    "tools.train_ppo_pfsp",
    "--total-timesteps",
    "200000",
    "--n-envs",
    "8",
    "--n-steps",
    "256",
    "--batch-size",
    "128",
    "--device",
    "cuda",
    "--warm-start",
    WARM_START,
    "--external-opponents",
    "agents/proxy/grid_il_lakhindar.py",
    "submissions/build_konbu_topk1/main.py",
    "submissions/build_rudra_topk1_proper/main.py",
    "--pool-max",
    "8",
    "--save-interval",
    "10000",
    "--pool-dir",
    POOL_DIR,
    "--self-play-prob",
    "0.6",
    "--external-prob",
    "0.3",
    "--reward-shaping",
    "--output",
    OUTPUT,
]
print("cmd:", " ".join(cmd))
with open(LOG, "w") as f:
    proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
print("exit:", proc.returncode, "log:", LOG)

## Cell 4: Checkpoint inventory + Drive push

完了後、 Drive 上に永続化された weight を確認。 user が local の `agents/proxy/ppo_v4_theta4.zip` に pull する。

In [ ]:
import glob
import os

print("=== Final pool inventory ===")
for p in sorted(glob.glob(f"{POOL_DIR}/*.zip")):
    size_mb = os.path.getsize(p) / 1024**2
    print(f"  {os.path.basename(p)}: {size_mb:.1f} MB")
print("=== Main output ===")
if os.path.exists(OUTPUT):
    print(f"  {OUTPUT}: {os.path.getsize(OUTPUT) / 1024**2:.1f} MB")
else:
    print(f"  MISSING: {OUTPUT}")
print("=== Next step ===")
print("1. User pulls {OUTPUT} to local agents/proxy/ppo_v4_theta4.zip")
print(
    "2. Run smoke test: uv run python -m tools._run_episode --left submissions/build_ppo_v4_theta4/main.py --right starter --p3 starter --p4 starter --seed 42"
)
print("3. Tournament gate: vs konbu17 24 ep 4P FFA, expect win_rate >= 0.5")